# Assignment 3: Approximation via Monte Carlo

This is the **Python (no-GenJAX)** stencil — numpy + scipy + matplotlib. If you'd prefer the GenJAX-first stencil, see `mc_approx.ipynb`. The math is identical; only the library calls differ. Each problem has a short *Now in GenJAX* note showing the one-line translation, in case you want to compare.

In this assignment you will explore three Monte Carlo methods:

1. **Problem 1** — compare **naive Monte Carlo** and **importance sampling** for estimating a tail probability $P(Y>2)$, $Y\sim N(0,1)$.
2. **Problem 2** — build a **Markov chain Monte Carlo** sampler (interleaving **Gibbs** and **Metropolis–Hastings**) for the hierarchical Beta-Binomial model of Kemp, Perfors & Tenenbaum (2007).
3. **Problem 3** — quantify how "efficient" a set of samples is with the **effective sample size** ($N_{\mathrm{eff}}$).

**Read `mc_approx.pdf` first** — it has the full problem statements and all the math. This notebook is the scaffold: cells marked `# fill me` are for you to complete. Submit the completed notebook (it must run end-to-end) *or* a single PDF report with your code, figures, and written answers.

**Textbook background:** Tutorial 3 Ch 12 (hierarchical Bayes / approximate inference). The Week 7 lecture covers Monte Carlo, importance sampling, and MCMC (Metropolis–Hastings + Gibbs).

In [ ]:
import numpy as np
from scipy.stats import norm, beta as Beta
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

# scipy cheatsheet:
#   rng.standard_normal(T)          -> standard normal draws
#   rng.beta(a, b)                  -> one Beta(a,b) draw
#   Beta.logpdf(x, a, b)            -> log density (vectorizes over x)
#   norm.logpdf(x, mu, sigma)       -> log density of N(mu, sigma^2)
# Now in GenJAX: rng.beta(a,b)->beta.sample(key,a,b); Beta.logpdf->beta.logpdf;
#                norm.logpdf->normal.logpdf.

---

## Problem 1: Monte Carlo vs. importance sampling

Estimate $P(Y>2)$ for $Y\sim N(0,1)$ two ways, with $T=1000$ samples.

- **(a) Naive MC.** Draw $x^{(1)},\dots,x^{(T)}\sim N(0,1)$ and form the cumulative estimate $f_{MC}(t) = \frac{1}{t}\sum_{s\le t} \mathbb{1}[x^{(s)}>2]$. Plot $f_{MC}$ vs. $t$. Report $f_{MC}(T)$ and explain the shape of the curve.
- **(b) Importance sampling.** Draw $x^{(1)},\dots,x^{(T)}\sim q=N(2,1)$ and form the cumulative IS estimate using the explicit weight $e^{-2x+2}\,\mathbb{1}[x>2]$. Plot $f_{IS}$ vs. $t$. Report $f_{IS}(T)$.
- **(c)** Compare the two curves: how and why do they differ? State a broad lesson about Monte Carlo approximation.

In [ ]:
# fill me -- Problem 1(a) naive MC and 1(b) importance sampling.
#
# (a) x = rng.standard_normal(T); ind = (x > 2); f_mc = np.cumsum(ind)/np.arange(1,T+1)
# (b) xis = 2 + rng.standard_normal(T); w = np.exp(-2*xis+2)*(xis>2); f_is = cumsum/arange
# Plot both vs t with the true value (1 - norm.cdf(2)) as a reference line.

T = 1000


---

## Problem 2: MCMC for the Kemp hierarchical Beta-Binomial model

You observe $M$ bags of marbles; bag $i$ shows $y_i$ white out of $n_i$ drawn. The model:
$$\theta_i \mid \kappa,\varphi \sim \mathrm{Beta}(\kappa\varphi,\ \kappa(1-\varphi)), \qquad y_i \mid n_i,\theta_i \sim \mathrm{Binomial}(\theta_i; n_i),$$
parameterized by the **mean** $\varphi=a/(a+b)$ and **concentration** $\kappa=a+b$ (so $a=\kappa\varphi$, $b=\kappa(1-\varphi)$). Priors: $\varphi\sim\mathrm{Uniform}(0,1)$ and a weak proper **log-normal** on $\kappa$, i.e. $\log\kappa\sim N(\mu_0,\sigma_0^2)$ with $\mu_0=\log 10$, $\sigma_0=1.5$.

The sampler does two moves per sweep:
- **Gibbs** resamples each $\theta_i$ from its conjugate posterior $\mathrm{Beta}(\kappa\varphi+y_i,\ \kappa(1-\varphi)+n_i-y_i)$.
- **Metropolis–Hastings** updates $(\varphi,\ell)$ with $\ell=\log\kappa$, via a symmetric Gaussian random walk. The acceptance ratio is the Beta likelihood of the current $\theta_i$'s times the log-normal prior ratio on $\ell$ (the proposal is symmetric, so there is no asymmetry-correction term, and there is no Jacobian — see the PDF).

See `mc_approx.pdf` Problem 2 for the full derivation and the acceptance formula.

*Now in GenJAX:* replace `rng.beta(a,b)` with `beta.sample(key,a,b)`, `Beta.logpdf` with `beta.logpdf`, and `norm.logpdf` with `normal.logpdf`; the loop is identical.

In [ ]:
MU0, S0 = np.log(10.0), 1.5

def ab(kappa, phi):
    return kappa * phi, kappa * (1 - phi)

bagset_I  = (np.array([9]*10),            np.array([20]*10))
bagset_II = (np.array([1]*5 + [19]*5),    np.array([20]*10))

In [ ]:
# fill me -- Problem 2(a): conjugate posterior over theta for fixed (kappa, phi).
# For each (kappa, phi) in the PDF, plot prior Beta(a,b) and the posteriors after
# (1,0), (5,5), (9,1) on one figure. Posterior = Beta(a + y, b + (n - y)).
grid = np.linspace(1e-3, 1-1e-3, 400)


In [ ]:
# fill me -- Problem 2(b)+(c): the Gibbs + MH sampler.
#
def sampler(y, n, T=3000, burn=500, s_phi=0.04, s_ell=0.30, seed=0):
    rng = np.random.default_rng(seed)
    M = len(y)
    phi, ell = 0.5, np.log(10.0)
    theta = np.full(M, 0.5)
    ks, ps = [], []
    n_acc = 0
    for t in range(T):
        kappa = np.exp(ell); a, b = ab(kappa, phi)
        # 1. GIBBS sweep: for each bag i (random order),
        #    theta[i] = rng.beta(a + y[i], b + (n[i] - y[i]))
        # 2. MH propose: phi_p = phi + s_phi*N(0,1); ell_p = ell + s_ell*N(0,1)
        # 3. accept iff 0<phi_p<1 and log(rng.uniform()) < log_c, where
        #    log_c =  sum(Beta.logpdf(theta, ap, bp)) - sum(Beta.logpdf(theta, a, b))
        #           + norm.logpdf(ell_p, MU0, S0)     - norm.logpdf(ell, MU0, S0)
        #    On reject, keep (phi, ell). Record kappa, phi after burn-in.
        # fill me
        pass
    return np.array(ks), np.array(ps), n_acc / T


In [ ]:
# fill me -- Problem 2(d): run on bag sets I and II; histograms of kappa and phi;
# report mean kappa, mean phi, and the predictive value (= mean phi).


In [ ]:
# fill me -- Problem 2(e): report acceptance rate per set; tune s_phi, s_ell to ~0.2-0.5;
# explain too-small vs too-large steps and why the two sets differ (connect to mixing).


---

## Problem 3: Effective sample size

The effective sample size summarizes how useful a set of weighted samples is:
$$N_{\mathrm{eff}} = \frac{1}{\sum_t (w^{(t)})^2}, \qquad w^{(t)}\ge 0,\ \textstyle\sum_t w^{(t)}=1.$$

- **(a) Good vs. bad proposal.** With target $p=N(0,1)$ and IS weights $w\propto p/q$, compute $N_{\mathrm{eff}}$ for a **good** proposal that overlaps $p$ well (e.g. $q=N(0,1.5^2)$) and a **bad** one that does not (e.g. $q=N(4,1)$). You should see good $q$ in the high hundreds, bad $q$ in the single digits. (Do **not** use $q=N(2,1)$ here — that is part (b).)
- **(b) The puzzle.** Compute $N_{\mathrm{eff}}$ for two estimators of $P(Y>2)$: the IS sampler with $q=N(2,1)$ ($w\propto p/q$), and plain MC from $p$ ($w=1/T$). MC reports $N_{\mathrm{eff}}=T=1000$ while IS reports ~50 — yet IS is the *more accurate* estimator (Problem 1). Resolve the puzzle in writing (see the PDF for the three points to address).

*Now in GenJAX:* the ESS formula is library-agnostic; only the weight computation changes (`norm.logpdf` -> `normal.logpdf`).

In [ ]:
# fill me -- Problem 3.
def ess(w):
    """Effective sample size from unnormalized nonnegative weights."""
    # fill me: normalize, then 1 / sum(w**2)
    pass

# (a) good q=N(0,1.5^2) vs bad q=N(4,1), target p=N(0,1), w = p/q. Report both ESS.
# (b) IS (q=N(2,1), w=p/q) vs plain MC (w=ones). Report both; write the resolution.
